# CEE-SD (Adaptive Tridecoding) 动态带宽调控分析

本笔记本分析了 `CEE-SD` (在实验代码中对应 `adaptive_tridecoding`) 如何根据网络带宽动态调整压缩参数 (Top-K) 和起草长度 (Draft Length)。

## 1. 导入数据分析库
首先导入必要的库，包括数据处理、可视化和科学计算工具。

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import pearsonr

# 设置绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 12

## 2. 加载并解析实验结果 JSON 文件
解析 `experiment_summary_20260203_215334.json` 文件。由于历史记录是对每个 step 的采样，我们需要将其展开。

In [ ]:
file_path = 'experiment_summary_20260203_215334.json'

with open(file_path, 'r') as f:
    data = json.load(f)

# 将记录展平
all_records = []
for entry in data:
    exp_name = entry['exp_name']
    res = entry['result']
    
    # 提取历史数据
    bw_hist = res.get('edge_cloud_bandwidth_history', [])
    topk_hist = res.get('edge_cloud_topk_history', [])
    draft_hist = res.get('edge_cloud_draft_len_history', [])
    
    # 确保长度一致（如果某些记录缺失，则忽略该条目）
    min_len = min(len(bw_hist), len(topk_hist), len(draft_hist))
    
    if min_len > 0:
        for i in range(min_len):
            all_records.append({
                'exp_name': exp_name,
                'algorithm': 'cee-sd' if 'adaptive_tridecoding' in exp_name else 'unknown',
                'step': i,
                'bandwidth': bw_hist[i],
                'topk': topk_hist[i],
                'draft_length': draft_hist[i]
            })

df = pd.DataFrame(all_records)
print(f"Total step records loaded: {len(df)}")
df.head()

## 3. 筛选并提取 cee-sd 核心数据
主要针对名为 `adaptive_tridecoding` 的算法进行分析。我们来看看不同带宽条件下的平均值。

In [ ]:
cee_sd_df = df[df['algorithm'] == 'cee-sd'].copy()

# 显示统计摘要
summary = cee_sd_df[['bandwidth', 'topk', 'draft_length']].describe()
print("CEE-SD 统计摘要:")
display(summary)

# 查看带宽的分位分布，检查是否有明显的变化
cee_sd_df['bandwidth_bin'] = pd.qcut(cee_sd_df['bandwidth'], q=5, duplicates='drop')
binned_avg = cee_sd_df.groupby('bandwidth_bin')[['topk', 'draft_length']].mean().reset_index()
print("\n不同带宽区间下的 Top-K 和 Draft Length 均值:")
display(binned_avg)

## 4. 分析带宽与 Top-K 的相关性
计算带宽与 `topk` 之间的 Pearson 相关系数，验证在低带宽时 `topk` 是否更小（即压缩倍率更高）。

In [ ]:
corr_topk, _ = pearsonr(cee_sd_df['bandwidth'], cee_sd_df['topk'])
print(f"带宽与 Top-K 的相关系数: {corr_topk:.4f}")

plt.figure(figsize=(10, 6))
sns.regplot(data=cee_sd_df, x='bandwidth', y='topk', scatter_kws={'alpha':0.3})
plt.title(f"Bandwidth vs Top-K (Correlation: {corr_topk:.4f})")
plt.xlabel("Bandwidth (Mbps)")
plt.ylabel("Top-K")
plt.show()

## 5. 分析带宽与起草长度的动态响应
研究起草长度 `draft_length` 如何随带宽波动。通常带宽增加时，更快的网络支持更大的起草长度。

In [ ]:
corr_draft, _ = pearsonr(cee_sd_df['bandwidth'], cee_sd_df['draft_length'])
print(f"带宽与 Draft Length 的相关系数: {corr_draft:.4f}")

plt.figure(figsize=(10, 6))
sns.regplot(data=cee_sd_df, x='bandwidth', y='draft_length', color='orange', scatter_kws={'alpha':0.3})
plt.title(f"Bandwidth vs Draft Length (Correlation: {corr_draft:.4f})")
plt.xlabel("Bandwidth (Mbps)")
plt.ylabel("Draft Length")
plt.show()

## 6. 带宽调控策略的可视化分析 (时间轴视角)
选取一个特定的实验任务，展示在推理过程中，系统如何根据带宽的波动实时调节参数。这将更直观地展示 `cee-sd` 的动态响应过程。

In [ ]:
# 选第一个实验作为示例
example_exp = cee_sd_df['exp_name'].unique()[0]
sample_df = cee_sd_df[cee_sd_df['exp_name'] == example_exp].sort_values('step')

fig, ax1 = plt.subplots(figsize=(14, 8))

# 第一个纵轴：带宽
color_bw = 'tab:blue'
ax1.set_xlabel('Step (Communication Event)')
ax1.set_ylabel('Bandwidth (Mbps)', color=color_bw)
ax1.plot(sample_df['step'], sample_df['bandwidth'], color=color_bw, linewidth=2, label='Bandwidth', alpha=0.8)
ax1.tick_params(axis='y', labelcolor=color_bw)

# 第二个纵轴：Top-K
ax2 = ax1.twinx()
color_topk = 'tab:red'
ax2.set_ylabel('Top-K', color=color_topk)
ax2.step(sample_df['step'], sample_df['topk'], color=color_topk, linestyle='--', linewidth=1.5, label='Top-K', where='post')
ax2.tick_params(axis='y', labelcolor=color_topk)

# 第三个纵轴：Draft Length (偏移位置)
ax3 = ax1.twinx()
ax3.spines.right.set_position(("axes", 1.15))
color_draft = 'tab:green'
ax3.set_ylabel('Draft Length', color=color_draft)
ax3.step(sample_df['step'], sample_df['draft_length'], color=color_draft, linestyle=':', linewidth=2, label='Draft Length', where='post')
ax3.tick_params(axis='y', labelcolor=color_draft)

plt.title(f"Dynamic Control Response for Experiment:\n{example_exp}")
fig.tight_layout()
plt.show()